# Task 2.1 — Dataset Selection and Setup (5 marks)

Choose a toy dataset for the PLTM reproduction, justify it, generate it, and document preprocessing.

**Do not clear outputs before submission.**

## Dataset choice: synthetic multifaceted toy data

**What the dataset is.**  
A custom synthetic dataset with **1,000 samples** and **4 features**. It is generated so that **two different ground-truth clusterings** exist in disjoint feature subspaces: **Facet 1** uses features 0 and 1 and is drawn from a **3-component** Gaussian mixture; **Facet 2** uses features 2 and 3 and is drawn from a **2-component** Gaussian mixture, independently of Facet 1. We store the data matrix **X** and the two label vectors (**labels_facet1**, **labels_facet2**) for evaluation.

**Why it is a reasonable testbed.**  
The paper’s central claim is that high-dimensional data can be *multifaceted*—i.e. several meaningful clusterings can exist on different subsets of variables. This dataset provides a clear ground truth for exactly two facets in two disjoint 2D subspaces, so we can test whether the PLTM (with a fixed two-pouch structure) recovers both latents and achieves high NMI with each facet’s labels. It is therefore well aligned with the paper’s problem type (model-based clustering with variable relevance / multiple facets).

**Limitations compared to the original paper.**  
The paper uses a 9-dimensional synthetic model (Figure 1(c)) with a more complex latent tree and dependencies between latents, and real UCI datasets (e.g. image, wine) with many more features and samples. Our toy data is lower-dimensional (4 features), has only two facets defined by hand, and the two facets are independent; structure and dependencies are not discovered by search but fixed in advance. It is intended to demonstrate the *core* idea (EM + eigenvalue constraints recovering multiple facets) within a 1.5–2 hour reproduction scope.

## Preprocessing

No preprocessing is applied beyond generation. The data are generated in the same scale (Gaussian components with comparable variances). If we later load from disk, we use the stored arrays as-is. Feature indices 0–1 and 2–3 correspond to Facet 1 and Facet 2 respectively.

## Generate the synthetic multifaceted dataset

We generate two facets independently, then combine into one 1,000 × 4 data matrix and two label vectors. Random seed is fixed for reproducibility.

In [1]:
import numpy as np
import os

# Reproducibility (documented at top of notebook)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

N_SAMPLES = 1000
N_FEATURES = 4

# --- Facet 1: 3 components, features 0 and 1 ---
# Means and covariances for 3-component bivariate GMM (well separated)
MEANS_FACET1 = np.array([
    [0.0,  0.0 ],
    [3.0,  0.0 ],
    [1.5,  2.5 ],
])
# Shared covariance per component (identity-like, small)
COV_FACET1 = np.array([[0.4, 0.1], [0.1, 0.4]])  # 2x2
N_COMPONENTS_FACET1 = 3
WEIGHTS_FACET1 = np.ones(N_COMPONENTS_FACET1) / N_COMPONENTS_FACET1

# --- Facet 2: 2 components, features 2 and 3 ---
MEANS_FACET2 = np.array([
    [0.0, 0.0],
    [2.5, 2.5],
])
COV_FACET2 = np.array([[0.35, 0.05], [0.05, 0.35]])
N_COMPONENTS_FACET2 = 2
WEIGHTS_FACET2 = np.ones(N_COMPONENTS_FACET2) / N_COMPONENTS_FACET2

def sample_facet(weights, means, cov, n_samples):
    """Sample from a Gaussian mixture: first draw component, then sample from that Gaussian."""
    n_comp = len(weights)
    comps = np.random.choice(n_comp, size=n_samples, p=weights)
    out = np.zeros((n_samples, means.shape[1]))
    for k in range(n_comp):
        mask = comps == k
        n_k = mask.sum()
        if n_k > 0:
            out[mask] = np.random.multivariate_normal(means[k], cov, size=n_k)
    return out, comps

# Generate Facet 1 (features 0, 1) and Facet 2 (features 2, 3) independently
X_facet1, labels_facet1 = sample_facet(WEIGHTS_FACET1, MEANS_FACET1, COV_FACET1, N_SAMPLES)
X_facet2, labels_facet2 = sample_facet(WEIGHTS_FACET2, MEANS_FACET2, COV_FACET2, N_SAMPLES)

# Combined data matrix: 1000 x 4
X = np.hstack([X_facet1, X_facet2])

print("Dataset shape:", X.shape)
print("Facet 1 labels (unique):", np.unique(labels_facet1))
print("Facet 2 labels (unique):", np.unique(labels_facet2))
print("Facet 1 label counts:", np.bincount(labels_facet1))
print("Facet 2 label counts:", np.bincount(labels_facet2))

Dataset shape: (1000, 4)
Facet 1 labels (unique): [0 1 2]
Facet 2 labels (unique): [0 1]
Facet 1 label counts: [346 339 315]
Facet 2 label counts: [519 481]


Save the dataset and labels to `partB/data/` so that Task 2.2 and 2.3 can load the same data. We save the matrix and label vectors as NumPy files.

In [2]:
# Save to partB/data/ for use in task 2 2 and task 2 3
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
np.save(os.path.join(DATA_DIR, "toy_multifacet_X.npy"), X)
np.save(os.path.join(DATA_DIR, "toy_multifacet_labels_facet1.npy"), labels_facet1)
np.save(os.path.join(DATA_DIR, "toy_multifacet_labels_facet2.npy"), labels_facet2)
print("Saved:", os.path.join(DATA_DIR, "toy_multifacet_X.npy"),
      "and label vectors (labels_facet1, labels_facet2).")

Saved: data/toy_multifacet_X.npy and label vectors (labels_facet1, labels_facet2).


In [3]:
# Quick verification: dataset meets requirements (≥100 samples, ≥2 features)
assert X.shape[0] >= 100 and X.shape[1] >= 2, "Dataset must have ≥100 samples and ≥2 features"
print("Dataset ready for Task 2.2. X shape:", X.shape, "| Facet 1 (3 clusters), Facet 2 (2 clusters).")

Dataset ready for Task 2.2. X shape: (1000, 4) | Facet 1 (3 clusters), Facet 2 (2 clusters).
